# National inventory levels (canonical products)

Cross-country **closing stock** panel (`CLOSTLV`) rolled up to `product_canonical`.

- **National sources only** (no JODI fallback).
- **Long-form CSV** for Tableau / PyGWalker: `data/processed/inventory/country_stocks_consolidated.csv`
- **Country registry**: `reference/inventory_sources.csv` — append a row when onboarding a new country.

> Workflow: run country `update_*.py` scripts first, then re-run the export cell (or `python scripts/export_inventory_consolidated.py`).

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "analytics").is_dir() and (p / "scripts").is_dir():
        ROOT = p
        break
else:
    raise RuntimeError("Run from country_oil_scraper/ or notebooks/")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analytics.inventory_consolidated import (
    build_consolidated_inventory,
    inventory_levels_table,
    load_inventory_sources,
    save_consolidated_csv,
    ytd_month_starts,
)

# --- knobs ---
YEAR = date.today().year
TARGET_UNIT = "mbbl"  # kb | mbbl | kt | ML | kL
#PRODUCT = "Kerosene"  # product_canonical from product_map.csv
AS_OF = date.today()  # YTD cutoff for column months
SHOW_TOTAL = True  # append a sum row across countries at the bottom

PROCESSED_DIR = ROOT / "data" / "processed"
CSV_PATH = PROCESSED_DIR / "inventory" / "country_stocks_consolidated.csv"
SOURCES_CSV = ROOT / "reference" / "inventory_sources.csv"

display(load_inventory_sources(SOURCES_CSV)[["country_key", "display_name", "notes"]])

,country_key,display_name,notes
0,italy,Italy,Demand-only MASE source — no CLOSTLV rows yet
1,spain,Spain,Demand-only CORES source — no CLOSTLV rows yet
2,korea,Korea,KNOC Petronet 석유제품재고 (native kb)
3,japan,Japan,METI 在庫 from 確報 (native kL/t)
4,taiwan,Taiwan,Demand-only MOEA source — no CLOSTLV rows yet
5,thailand,Thailand,Demand-only EPPO source — no CLOSTLV rows yet
6,australia,Australia,DCCEEW stock volume by product (native ML etc.)
7,portugal,Portugal,Demand-only dgeg source no CLOSTLV
8,uk,United Kingdom,DESNZ ET 3.11 product stocks (native kt)
9,hungary,Hungary,MEKH HaviOlajKeszlet CSNATTER (native kt)


## 1. Build & export consolidated CSV

Long format: one row per `(country, month, product_canonical)`. Includes `value_kb` (fixed scale) plus `value`/`unit` for the selected display unit.

In [2]:
consolidated = build_consolidated_inventory(
    processed_dir=PROCESSED_DIR,
    sources_csv=SOURCES_CSV,
)

if consolidated.empty:
    print("[warn] No CLOSTLV rows in any registered parquet — CSV will be empty.")
else:
    print(
        f"Loaded {len(consolidated):,} rows  "
        f"{consolidated['date'].min().date()} → {consolidated['date'].max().date()}"
    )
    print(
        "Countries with data:",
        sorted(consolidated["country"].unique()),
    )

saved = save_consolidated_csv(consolidated, CSV_PATH, target_unit=TARGET_UNIT)
print(f"Wrote {saved}")
consolidated.head(10)

Loaded 7,125 rows  1992-01-01 → 2026-05-01
Countries with data: ['Australia', 'Hungary', 'Japan', 'Korea', 'Ukraine', 'United Kingdom']
Wrote c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\processed\inventory\country_stocks_consolidated.csv


,country_key,country,date,product_canonical,metric_type,value_native,unit_native,value_kb,is_provisional,source,year,month,month_label
0,australia,Australia,2010-07-01,Diesel,CLOSTLV,938.6,ML,5903.615666,False,dceew_petroleum_statistics,2010,7,Jul
1,australia,Australia,2010-08-01,Diesel,CLOSTLV,906.0,ML,5698.567860,False,dceew_petroleum_statistics,2010,8,Aug
2,australia,Australia,2010-09-01,Diesel,CLOSTLV,1047.7,ML,6589.833937,False,dceew_petroleum_statistics,2010,9,Sep
3,australia,Australia,2010-10-01,Diesel,CLOSTLV,959.4,ML,6034.443714,False,dceew_petroleum_statistics,2010,10,Oct
4,australia,Australia,2010-11-01,Diesel,CLOSTLV,943.8,ML,5936.322678,False,dceew_petroleum_statistics,2010,11,Nov
5,australia,Australia,2010-12-01,Diesel,CLOSTLV,934.2,ML,5875.940502,False,dceew_petroleum_statistics,2010,12,Dec
6,australia,Australia,2011-01-01,Diesel,CLOSTLV,979.3,ML,6159.610933,False,dceew_petroleum_statistics,2011,1,Jan
7,australia,Australia,2011-02-01,Diesel,CLOSTLV,914.4,ML,5751.402264,False,dceew_petroleum_statistics,2011,2,Feb
8,australia,Australia,2011-03-01,Diesel,CLOSTLV,941.8,ML,5923.743058,False,dceew_petroleum_statistics,2011,3,Mar
9,australia,Australia,2011-04-01,Diesel,CLOSTLV,1079.5,ML,6789.849895,False,dceew_petroleum_statistics,2011,4,Apr


In [3]:
consolidated['product_canonical'].unique()

<StringArray>
[             'Diesel',            'Fuel Oil',            'Gasoline',
            'Jet Fuel',                 'LPG', 'Lubricants / Grease',
              'Others',             'Naphtha',             'Petcoke',
             'Bitumen',              'Gasoil',              'Grease',
            'Kerosene',          'Lubricants',                 'Wax']
Length: 15, dtype: string

## 2. YTD inventory matrix (country × month)

Matches the spreadsheet layout: rows = all registered countries, columns = Jan … current month. Missing product or unreported month → `N/a`.

In [ ]:
PRODUCT = "Gasoline"
ytd_months = ytd_month_starts(YEAR, as_of=AS_OF)
print(f"YTD months for {YEAR} (through {AS_OF}): {[m.strftime('%b') for m in ytd_months]}")

table = inventory_levels_table(
    consolidated,
    product_canonical=PRODUCT,
    year=YEAR,
    target_unit=TARGET_UNIT,
    as_of=AS_OF,
    missing_label="-",
    include_total=SHOW_TOTAL,
    sources_csv=SOURCES_CSV,
)

print(f"Inventory levels — {PRODUCT} ({TARGET_UNIT})")
display(table)

YTD months for 2026 (through 2026-06-26): ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
Inventory levels — Gasoline (mbbl)


,Jan,Feb,Mar,Apr,May,Jun
Italy,-,-,-,-,-,-
Spain,-,-,-,-,-,-
Korea,5.756,5.096,6.743,7.02,5.873,-
Japan,10.734,10.954,10.237,11.373,-,-
Taiwan,-,-,-,-,-,-
Thailand,-,-,-,-,-,-
Australia,7.924,7.738,8.789,9.889,-,-
Portugal,-,-,-,-,-,-
United Kingdom,8.528,8.405,7.716,-,-,-
Hungary,5.158,5.184,4.606,-,-,-


: 

## Adding a new country

1. Wire up scraper + processor so the parquet has `metric_type = CLOSTLV` rows with `product_canonical`.
2. Append one row to `reference/inventory_sources.csv` (`parquet_subdir`, `parquet_filename`, `sort_order`).
3. Re-run this notebook or `python scripts/export_inventory_consolidated.py`.

No code changes required unless the new source needs a custom unit conversion (today only Japan METI is special-cased).